# Culturomics — Method 6: Suppression & Endogenous Censorship
### Replication and Extension of Michel et al. (2011) §6 using Wikipedia "Cancel Culture" Cases (2001–2024)

**Course:** Computational Linguistics — Ca' Foscari University of Venice
**Original paper:** Michel, J.-B. et al. (2011). *Quantitative Analysis of Culture Using Millions
of Digitized Books*. Science, 331(6014), 176–182.

---

## Conceptual Framework

Michel et al. (2011) detected **external state censorship** by measuring the suppression index:

$$s = \frac{\text{freq}(\text{contested period})}{\text{freq}(\text{baseline})}$$

where *s* < 1 indicates suppression and *s* > 1 indicates amplification. Their key finding:
in Nazi Germany (1933–1945), 9.8% of individuals showed strong suppression (*s* < 1/5),
and this population was highly enriched in documented victims of repression.

### Our Extension: Endogenous Suppression in Wikipedia

Wikipedia offers a fundamentally different case: **endogenous community censorship**,
where editors — not governments — collectively suppress certain names after public
"cancellation" events. This is measurable through two independent signals:

1. **Frequency signal** (same as Michel et al.): does the person's name appear less often
   in Wikipedia articles *after* their cancellation event?
2. **Edit activity signal** (our original contribution, not in Michel et al.): does the
   number of edits to their article *spike* during the contested period, suggesting
   active editorial conflict over how to represent the person?

The combination of *low frequency + high edit count* is a signature of **active suppression**,
distinguishing it from mere irrelevance (low frequency + low edit count).

### Cases Studied

Seven individuals whose "cancellation" events have precise, publicly documented dates,
allowing us to define clear baseline and contested periods:

| Person | Scandal date | Nature | Contested period |
|---|---|---|---|
| Harvey Weinstein | Oct 2017 | #MeToo sexual assault | 2017–2019 |
| Kevin Spacey | Oct 2017 | #MeToo sexual assault | 2017–2019 |
| J.K. Rowling | Jun 2020 | Trans rights controversy | 2020–2024 |
| Louis C.K. | Nov 2017 | #MeToo sexual misconduct | 2017–2020 |
| Johnny Depp | 2016 | Amber Heard abuse claims | 2016–2022 |
| Roseanne Barr | May 2018 | Racist tweet, fired | 2018–2020 |
| R. Kelly | Jan 2019 | *Surviving R. Kelly* | 2019–2022 |


## 0. Setup

In [ ]:
import requests
import time
import re
import json
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#FAFAFA',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    10,
    'axes.labelsize':    8.5,
    'xtick.labelsize':   7.5,
    'ytick.labelsize':   7.5,
})

PRIMARY   = '#C0392B'
SECONDARY = '#2C3E50'
ACCENT    = '#E67E22'
GREEN     = '#27AE60'
PURPLE    = '#8E44AD'
BLUE      = '#2980B9'

API_URL = "https://en.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "CulturomicsEssayBot/1.0 (academic research)"}

OUTPUT_DIR = Path("culturomics_method6_output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Setup complete.")

## 1. Case Definitions

Each case has:
- `article`: exact Wikipedia page title
- `scandal_year`: year of the primary cancellation event
- `baseline`: (start_year, end_year) — period before scandal, our "before" window
- `contested`: (start_year, end_year) — period of suppression to measure
- `search_terms`: name variants to count in *other* articles (cross-article signal)

**Methodological note:** unlike Michel et al., who measured name frequency
across the *entire* corpus, we measure two things:
1. Frequency within the person's **own Wikipedia article** (how their page changes)
2. Frequency of their name in the **broader corpus** of 80 articles (cross-article signal)

This dual measurement lets us distinguish *page-level* editorial control
from *corpus-level* cultural suppression.


In [ ]:
CASES = {
    "Harvey_Weinstein": {
        "display":      "Harvey Weinstein",
        "scandal_year": 2017,
        "baseline":     (2010, 2016),
        "contested":    (2017, 2020),
        "post":         (2021, 2024),
        "search_terms": ["weinstein"],
        "color":        PRIMARY,
    },
    "Kevin_Spacey": {
        "display":      "Kevin Spacey",
        "scandal_year": 2017,
        "baseline":     (2010, 2016),
        "contested":    (2017, 2020),
        "post":         (2021, 2024),
        "search_terms": ["spacey"],
        "color":        SECONDARY,
    },
    "J._K._Rowling": {
        "display":      "J.K. Rowling",
        "scandal_year": 2020,
        "baseline":     (2010, 2019),
        "contested":    (2020, 2022),
        "post":         (2023, 2024),
        "search_terms": ["rowling"],
        "color":        ACCENT,
    },
    "Louis_C.K.": {
        "display":      "Louis C.K.",
        "scandal_year": 2017,
        "baseline":     (2010, 2016),
        "contested":    (2017, 2020),
        "post":         (2021, 2024),
        "search_terms": ["louis", "ck"],
        "color":        GREEN,
    },
    "Johnny_Depp": {
        "display":      "Johnny Depp",
        "scandal_year": 2016,
        "baseline":     (2005, 2015),
        "contested":    (2016, 2022),
        "post":         (2023, 2024),
        "search_terms": ["depp"],
        "color":        PURPLE,
    },
    "Roseanne_Barr": {
        "display":      "Roseanne Barr",
        "scandal_year": 2018,
        "baseline":     (2010, 2017),
        "contested":    (2018, 2020),
        "post":         (2021, 2024),
        "search_terms": ["roseanne", "barr"],
        "color":        BLUE,
    },
    "R._Kelly": {
        "display":      "R. Kelly",
        "scandal_year": 2019,
        "baseline":     (2010, 2018),
        "contested":    (2019, 2022),
        "post":         (2023, 2024),
        "search_terms": ["kelly"],
        "color":        "#E74C3C",
    },
}

OBS_YEARS  = list(range(2001, 2025))
CACHE_FILE = OUTPUT_DIR / "revision_cache_m6.json"

print(f"Cases defined: {len(CASES)}")
for name, cfg in CASES.items():
    print(f"  {cfg['display']}: scandal {cfg['scandal_year']}, "
          f"baseline {cfg['baseline']}, contested {cfg['contested']}")

## 2. Signal 1 — Text Frequency (Replication of Michel et al.)

Fetch yearly snapshots of each person's Wikipedia article and measure
how often their own name appears in the text — the same 1-gram frequency
method Michel et al. used for Marc Chagall in German books.


In [ ]:
def clean_wikitext(raw: str) -> str:
    text = re.sub(r'\{\{[^}]*\}\}', ' ', raw)
    text = re.sub(r'\[\[(?:File|Image):[^\]]*\]\]', ' ', text, flags=re.I)
    text = re.sub(r'\[\[(?:[^|\]]*\|)?([^\]]+)\]\]', r'\1', text)
    text = re.sub(r'\[https?://\S+[^\]]*\]', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'={2,}[^=]+=+', ' ', text)
    text = re.sub(r"'{2,}", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def fetch_snapshot(article: str, year: int) -> str:
    """Get Wikipedia article text closest to mid-January of `year`."""
    params = {
        "action": "query", "titles": article, "prop": "revisions",
        "rvprop": "ids|timestamp", "rvlimit": 1,
        "rvstart": f"{year}-01-20T00:00:00Z",
        "rvend":   f"{year}-01-01T00:00:00Z",
        "rvdir":   "older", "format": "json",
    }
    try:
        r = requests.get(API_URL, params=params, headers=HEADERS, timeout=20)
        pages = r.json().get("query", {}).get("pages", {})
        page  = list(pages.values())[0]
        revs  = page.get("revisions", [])
        if not revs: return ""
        rev_id = revs[0]["revid"]
        r2 = requests.get(API_URL, headers=HEADERS, timeout=30, params={
            "action": "query", "revids": rev_id, "prop": "revisions",
            "rvprop": "content", "rvslots": "main", "format": "json",
        })
        pages2 = r2.json().get("query", {}).get("pages", {})
        raw = list(pages2.values())[0]["revisions"][0]["slots"]["main"]["*"]
        return clean_wikitext(raw)
    except Exception:
        return ""


def build_text_corpus(cases: dict, years: list, cache_file: Path) -> dict:
    """Fetch and cache yearly text for each person's Wikipedia article."""
    if cache_file.exists():
        with open(cache_file, encoding='utf-8') as f:
            corpus = json.load(f)
        print(f"Loaded cache: {sum(len(v) for v in corpus.values())} entries")
    else:
        corpus = {}

    total = len(cases) * len(years)
    done  = sum(len(v) for v in corpus.values())

    for article in cases:
        if article not in corpus:
            corpus[article] = {}
        for year in years:
            yk = str(year)
            if yk in corpus[article]:
                continue
            corpus[article][yk] = fetch_snapshot(article, year)
            done += 1
            if done % 15 == 0:
                print(f"  Progress: {done}/{total} ({done/total*100:.0f}%)")
                with open(cache_file, 'w', encoding='utf-8') as f:
                    json.dump(corpus, f, ensure_ascii=False)
            time.sleep(0.5)

    with open(cache_file, 'w', encoding='utf-8') as f:
        json.dump(corpus, f, ensure_ascii=False)
    print("Text corpus ready.")
    return corpus


def tokenize(text: str) -> list:
    return re.findall(r'\b[a-z]+\b', text.lower())


def name_freq_in_text(text: str, search_terms: list) -> float:
    """Relative frequency of any search term in text."""
    tokens = tokenize(text)
    if not tokens: return 0.0
    count = sum(tokens.count(t.lower()) for t in search_terms)
    return count / len(tokens)


print("Functions defined.")

In [ ]:
# ── FETCH TEXT CORPUS (Signal 1) ─────────────────────────────
text_corpus = build_text_corpus(CASES, OBS_YEARS, CACHE_FILE)
print("\nSample check — Harvey Weinstein 2016 text length:",
      len(text_corpus.get("Harvey_Weinstein", {}).get("2016", "")), "chars")
print("Sample check — Harvey Weinstein 2018 text length:",
      len(text_corpus.get("Harvey_Weinstein", {}).get("2018", "")), "chars")

## 3. Signal 2 — Wikipedia Edit Count (Our Original Contribution)

Michel et al. could only measure the *outcome* of censorship (name disappears from text).
Wikipedia's revision history lets us also measure the *process* — how much editorial
conflict occurred around each person's article in each year.

**Hypothesis:** if suppression is genuine (not just declining relevance), we expect:
- **Baseline**: normal edit rate
- **Contested period**: spike in edits (editors fighting over how to represent the person)
- **Post period**: edit rate returns to (lower) normal, but text frequency stays suppressed

This combination — high edits + low frequency — is the Wikipedia signature of
**active suppression** as distinct from mere irrelevance.

The edit count is fetched via the MediaWiki revisions API, counting all revisions
per article per calendar year.


In [ ]:
def fetch_edit_count_by_year(article: str, year: int) -> int:
    """Count total number of edits to an article in a given calendar year.
    Uses rvlimit=500 per call with continuation — sums all revisions."""
    total_edits = 0
    params = {
        "action":  "query",
        "titles":  article,
        "prop":    "revisions",
        "rvprop":  "timestamp",
        "rvlimit": 500,
        "rvstart": f"{year}-12-31T23:59:59Z",
        "rvend":   f"{year}-01-01T00:00:00Z",
        "rvdir":   "older",
        "format":  "json",
    }
    while True:
        try:
            r    = requests.get(API_URL, params=params, headers=HEADERS, timeout=25)
            data = r.json()
            pages = data.get("query", {}).get("pages", {})
            page  = list(pages.values())[0]
            revs  = page.get("revisions", [])
            total_edits += len(revs)
            if "continue" in data:
                params["rvcontinue"] = data["continue"]["rvcontinue"]
                time.sleep(0.3)
            else:
                break
        except Exception:
            break
    return total_edits


def build_edit_corpus(cases: dict, years: list) -> dict:
    """Fetch and cache yearly edit counts. Stored separately from text corpus."""
    edit_cache_file = OUTPUT_DIR / "edit_counts_cache.json"

    if edit_cache_file.exists():
        with open(edit_cache_file, encoding='utf-8') as f:
            edit_counts = json.load(f)
        print(f"Loaded edit count cache: {sum(len(v) for v in edit_counts.values())} entries")
    else:
        edit_counts = {}

    total = len(cases) * len(years)
    done  = sum(len(v) for v in edit_counts.values())

    for article in cases:
        if article not in edit_counts:
            edit_counts[article] = {}
        for year in years:
            yk = str(year)
            if yk in edit_counts[article]:
                continue
            count = fetch_edit_count_by_year(article, year)
            edit_counts[article][yk] = count
            done += 1
            if done % 10 == 0:
                print(f"  Edit counts: {done}/{total} ({done/total*100:.0f}%)")
                with open(edit_cache_file, 'w') as f:
                    json.dump(edit_counts, f)
            time.sleep(0.6)

    with open(edit_cache_file, 'w') as f:
        json.dump(edit_counts, f)
    print("Edit count corpus ready.")
    return edit_counts


print("Edit count functions defined.")

In [ ]:
# ── FETCH EDIT COUNTS (Signal 2) ─────────────────────────────
# Note: this takes ~15-20 minutes for 7 cases × 24 years
# Progress is saved every 10 entries — safe to interrupt and restart

edit_counts = build_edit_corpus(CASES, OBS_YEARS)

# Quick sanity check
print("\nSanity check — expected: Weinstein edits spike in 2017/2018")
wstein_edits = edit_counts.get("Harvey_Weinstein", {})
for yr in [2015, 2016, 2017, 2018, 2019, 2020]:
    print(f"  {yr}: {wstein_edits.get(str(yr), 'N/A')} edits")

## 4. Suppression Index — Exact Replication of Michel et al.

$$s = \frac{\text{mean freq}(\text{contested period})}{\text{mean freq}(\text{baseline})}$$

- *s* < 1 → suppression (name appears less often after scandal)
- *s* > 1 → amplification (name becomes more prominent)
- *s* ≈ 0.2 → strong suppression (Michel et al.'s threshold for Nazi victims)

We also compute an **edit index** (our extension):

$$e = \frac{\text{mean edits}(\text{contested period})}{\text{mean edits}(\text{baseline})}$$

- *e* > 1 + suppression (*s* < 1) → **active suppression** (editors fighting over content)
- *e* ≈ 1 + suppression (*s* < 1) → **passive suppression** (declining relevance, not active editing)


In [ ]:
rows = []

for article, cfg in CASES.items():
    base_start, base_end   = cfg['baseline']
    cont_start, cont_end   = cfg['contested']
    post_start, post_end   = cfg['post']
    terms = cfg['search_terms']

    def mean_freq(start, end):
        freqs = []
        for yr in range(start, end + 1):
            text = text_corpus.get(article, {}).get(str(yr), "")
            freqs.append(name_freq_in_text(text, terms))
        return np.mean(freqs) if freqs else 0.0

    def mean_edits(start, end):
        ec = edit_counts.get(article, {})
        vals = [ec.get(str(yr), 0) for yr in range(start, end + 1)]
        return np.mean(vals) if vals else 0.0

    freq_base = mean_freq(base_start, base_end)
    freq_cont = mean_freq(cont_start, cont_end)
    freq_post = mean_freq(post_start, post_end)

    edit_base = mean_edits(base_start, base_end)
    edit_cont = mean_edits(cont_start, cont_end)
    edit_post = mean_edits(post_start, post_end)

    # Suppression index (Michel et al. formula)
    s_contested = freq_cont / freq_base if freq_base > 0 else float('nan')
    s_post      = freq_post / freq_base if freq_base > 0 else float('nan')

    # Edit index (our extension)
    e_contested = edit_cont / edit_base if edit_base > 0 else float('nan')
    e_post      = edit_post / edit_base if edit_base > 0 else float('nan')

    # Classification
    if s_contested < 0.5 and e_contested > 1.5:
        suppression_type = "ACTIVE suppression"
    elif s_contested < 0.5 and e_contested <= 1.5:
        suppression_type = "PASSIVE suppression"
    elif s_contested >= 1.0:
        suppression_type = "AMPLIFICATION"
    else:
        suppression_type = "mild suppression"

    rows.append({
        "name":          cfg['display'],
        "scandal_year":  cfg['scandal_year'],
        "freq_base":     round(freq_base * 10000, 4),
        "freq_cont":     round(freq_cont * 10000, 4),
        "freq_post":     round(freq_post * 10000, 4),
        "s_contested":   round(s_contested, 3),
        "s_post":        round(s_post, 3),
        "edit_base":     round(edit_base, 1),
        "edit_cont":     round(edit_cont, 1),
        "edit_post":     round(edit_post, 1),
        "e_contested":   round(e_contested, 3),
        "e_post":        round(e_post, 3),
        "type":          suppression_type,
        "color":         cfg['color'],
    })

df_sup = pd.DataFrame(rows)
print(df_sup[['name', 's_contested', 'e_contested', 'type']].to_string(index=False))

## 5. Visualisation

### 5-panel figure mirroring Michel et al. Figure 4


In [ ]:
fig = plt.figure(figsize=(18, 16))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.48, wspace=0.35)

# ── Panel A: Frequency trajectories (mirrors Michel Fig 4A — Chagall) ──
ax1 = fig.add_subplot(gs[0, :])   # full-width top panel

for article, cfg in CASES.items():
    terms = cfg['search_terms']
    freqs = []
    for yr in OBS_YEARS:
        text = text_corpus.get(article, {}).get(str(yr), "")
        freqs.append(name_freq_in_text(text, terms) * 10000)
    ax1.plot(OBS_YEARS, freqs, lw=2, color=cfg['color'],
             alpha=0.85, label=cfg['display'])
    # Mark scandal year
    scandal_idx = OBS_YEARS.index(cfg['scandal_year'])
    ax1.axvline(cfg['scandal_year'], color=cfg['color'],
                lw=0.8, ls=':', alpha=0.5)
    ax1.annotate("▼", (cfg['scandal_year'], freqs[scandal_idx]),
                 fontsize=9, color=cfg['color'], ha='center', va='bottom')

ax1.set_title("A. Name Frequency in Wikipedia Article Over Time
"
              "(analogous to Michel et al. Fig. 4A — Marc Chagall in German books)
"
              "▼ marks year of 'cancellation' event",
              fontweight='bold')
ax1.set_xlabel("Year")
ax1.set_ylabel("Relative frequency (per 10,000 tokens)")
ax1.legend(fontsize=8, ncol=4, loc='upper left')

# ── Panel B: Suppression index scatter (mirrors Michel Fig 4F) ──
ax2 = fig.add_subplot(gs[1, 0])

sc = ax2.scatter(df_sup['s_contested'], df_sup['e_contested'],
                 c=df_sup['color'].tolist(), s=200, zorder=4, alpha=0.9)

for _, row in df_sup.iterrows():
    ax2.annotate(row['name'].split()[-1],
                 (row['s_contested'], row['e_contested']),
                 textcoords="offset points", xytext=(7, 4),
                 fontsize=7.5, color=row['color'], fontweight='bold')

# Quadrant lines
ax2.axvline(1.0, color='gray', lw=1, ls='--', alpha=0.5)
ax2.axhline(1.0, color='gray', lw=1, ls='--', alpha=0.5)

# Michel et al. threshold: s < 0.2 = strong suppression
ax2.axvline(0.2, color=PRIMARY, lw=1.2, ls=':', alpha=0.7,
            label='Michel et al. strong suppression threshold (s=0.2)')

# Quadrant labels
ax2.text(0.1, ax2.get_ylim()[1]*0.95 if ax2.get_ylim()[1] > 0 else 2,
         "ACTIVE
SUPPRESSION", fontsize=7.5, color=PRIMARY,
         fontweight='bold', ha='center', va='top')
ax2.text(1.5, ax2.get_ylim()[1]*0.95 if ax2.get_ylim()[1] > 0 else 2,
         "AMPLIFICATION
+ conflict", fontsize=7.5, color=GREEN,
         ha='center', va='top')

ax2.set_title("B. Suppression Index (s) vs. Edit Index (e)
"
              "Our extension: s = freq(contested)/freq(baseline), e = edits(contested)/edits(baseline)",
              fontweight='bold')
ax2.set_xlabel("Suppression index s  (< 1 = suppressed, < 0.2 = strongly suppressed)")
ax2.set_ylabel("Edit index e  (> 1 = more edits during scandal)")
ax2.legend(fontsize=7, loc='lower right')

# ── Panel C: Frequency before/during/after comparison ──────
ax3 = fig.add_subplot(gs[1, 1])

x   = np.arange(len(df_sup))
w   = 0.25
ax3.bar(x - w, df_sup['freq_base'],  width=w, label='Baseline',   alpha=0.8, color=GREEN)
ax3.bar(x,     df_sup['freq_cont'],  width=w, label='Contested',  alpha=0.8, color=PRIMARY)
ax3.bar(x + w, df_sup['freq_post'],  width=w, label='Post',       alpha=0.8, color=ACCENT)
ax3.set_xticks(x)
ax3.set_xticklabels([r['name'].split()[-1] for _, r in df_sup.iterrows()],
                    rotation=30, ha='right', fontsize=8)
ax3.set_title("C. Mean Frequency per Period
(per 10k tokens: baseline / contested / post)",
              fontweight='bold')
ax3.set_ylabel("Mean relative frequency (×10⁴)")
ax3.legend(fontsize=8)

# ── Panel D: Edit counts before/during/after ────────────────
ax4 = fig.add_subplot(gs[2, 0])

ax4.bar(x - w, df_sup['edit_base'],  width=w, label='Baseline',  alpha=0.8, color=GREEN)
ax4.bar(x,     df_sup['edit_cont'],  width=w, label='Contested', alpha=0.8, color=PRIMARY)
ax4.bar(x + w, df_sup['edit_post'],  width=w, label='Post',      alpha=0.8, color=ACCENT)
ax4.set_xticks(x)
ax4.set_xticklabels([r['name'].split()[-1] for _, r in df_sup.iterrows()],
                    rotation=30, ha='right', fontsize=8)
ax4.set_title("D. Mean Annual Edit Count per Period
(our original contribution — not in Michel et al.)",
              fontweight='bold')
ax4.set_ylabel("Mean edits per year")
ax4.legend(fontsize=8)

# ── Panel E: Suppression index distribution (mirrors Michel Fig 4F histogram) ──
ax5 = fig.add_subplot(gs[2, 1])

s_vals = df_sup['s_contested'].dropna().values
ax5.hist(s_vals, bins=10, color=PRIMARY, alpha=0.7,
         edgecolor='white', linewidth=0.8)
ax5.axvline(1.0, color=SECONDARY, lw=2, ls='--', label='s = 1 (no change)')
ax5.axvline(0.2, color=ACCENT, lw=2, ls=':', label='s = 0.2 (Michel et al. strong suppression)')

for s_val, name in zip(df_sup['s_contested'], df_sup['name']):
    ax5.annotate(name.split()[-1],
                 (s_val, 0.1),
                 textcoords="offset points", xytext=(0, 5),
                 fontsize=6.5, rotation=90, ha='center',
                 color=PRIMARY)

ax5.set_title("E. Distribution of Suppression Indices
"
              "(mirrors Michel et al. Fig. 4F — German vs. English corpus distribution)",
              fontweight='bold')
ax5.set_xlabel("Suppression index s  (contested / baseline)")
ax5.set_ylabel("Count of individuals")
ax5.legend(fontsize=8)

# Main title
fig.suptitle(
    "Method 6 — Endogenous Suppression in Wikipedia: Cancel Culture Cases (2001–2024)
"
    "Replication and Extension of Michel et al. (2011) Figure 4",
    fontsize=13, fontweight='bold', y=1.01, color=SECONDARY
)

plt.savefig(OUTPUT_DIR / "method6_suppression.png",
            dpi=160, bbox_inches='tight', facecolor='white')
plt.show()
print("Figure saved to:", OUTPUT_DIR / "method6_suppression.png")

## 6. Direct Comparison with Michel et al. (2011)

The table below mirrors the implicit comparison in Michel et al. §6,
adding our Wikipedia findings as a third column.


In [ ]:
print("=" * 75)
print("SUPPRESSION COMPARISON: Michel et al. (1933-1945) vs. Wikipedia Cancel Culture")
print("=" * 75)

print("\nMichel et al. key findings (Nazi Germany, German book corpus):")
print("  Artists suppressed:     -56% (s ≈ 0.44)")
print("  Philosophers:           -76% (s ≈ 0.24)")
print("  Politicians:            -60% (s ≈ 0.40)")
print("  Strong suppression:      9.8% of individuals showed s < 0.2")
print("  Nazi party members:     +500% increase (s ≈ 6.0)")

print("\nOur Wikipedia findings (cancel culture, 2001-2024):")
print(f"{'Name':<20} {'s (contested)':<18} {'e (edits)':<15} {'Type'}")
print("-" * 70)
for _, row in df_sup.sort_values('s_contested').iterrows():
    print(f"{row['name']:<20} {row['s_contested']:<18} {row['e_contested']:<15} {row['type']}")

strong_sup = df_sup[df_sup['s_contested'] < 0.5]
amplified  = df_sup[df_sup['s_contested'] > 1.0]
print(f"\nStrongly suppressed (s < 0.5): {len(strong_sup)}/{len(df_sup)} cases")
print(f"Amplified (s > 1.0): {len(amplified)}/{len(df_sup)} cases")

print("\nKey difference from Michel et al.:")
print("  External censorship (Nazi Germany): s drops because STATE bans mention.")
print("  Endogenous suppression (Wikipedia): s may DROP or RISE depending on case.")
print("  The edit index (e) disambiguates:")
print("  → e > 1 + s < 1 = ACTIVE suppression (editors removing content)")
print("  → e > 1 + s > 1 = AMPLIFICATION through controversy")
print("  → e ≈ 1 + s < 1 = PASSIVE decline (person simply less relevant)")

## 7. Key Findings for the Essay

Copy these numbers directly into your essay.


In [ ]:
print("=" * 65)
print("METHOD 6 — ESSAY-READY SUMMARY")
print("=" * 65)
print()
print("Corpus: Wikipedia articles of 7 'cancelled' individuals, 2001–2024")
print()

for _, row in df_sup.sort_values('s_contested').iterrows():
    print(f"  {row['name']}")
    print(f"    Suppression index s = {row['s_contested']:.3f}  "
          f"({'↓ suppressed' if row['s_contested'] < 1 else '↑ amplified'})")
    print(f"    Edit index e = {row['e_contested']:.3f}  "
          f"({'↑ more conflict' if row['e_contested'] > 1 else '↓ less editing'})")
    print(f"    Classification: {row['type']}")
    print()

print("For context — Michel et al. (2011) thresholds:")
print("  s < 0.2  = strong suppression (Nazi victims: 9.8% of individuals)")
print("  s > 5.0  = strong amplification (Nazi party members: 1.5% of individuals)")
print()
print("Key 'and so what' argument:")
print("  Michel et al. assumed suppression index s < 1 always means censorship.")
print("  Wikipedia data shows this is insufficient: s < 1 with e > 1 means")
print("  active editorial suppression; s < 1 with e ≈ 1 means passive irrelevance.")
print("  The edit index is a new dimension of culturomics analysis that")
print("  distinguishes these two mechanisms — impossible with static print corpora.")

df_sup.drop(columns=['color']).to_csv(OUTPUT_DIR / "method6_suppression_results.csv", index=False)
print(f"\nResults saved to: {OUTPUT_DIR}/method6_suppression_results.csv")